In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [9]:
# --- CARGAR LOS DATOS LIMPIOS ---
df = pd.read_csv("Dataset_ALDIMI_Merged_Clean.csv")

print("--- DATOS ORIGINALES ---")
print(f"Forma del dataset: {df.shape}")
print(df.head(3))
print("\nInformación del dataset:")
df.info()

--- DATOS ORIGINALES ---
Forma del dataset: (3000, 24)
   Patient_ID Cancer_Type  Age  Gender  Smoking  Alcohol_Use  Obesity  \
0        1001        Lung   52       0       10            7        4   
1        1002       Colon   73       1        5            4        8   
2        1003        Skin   70       0        3            0        2   

   Family_History  Diet_Red_Meat  Diet_Salted_Processed  ...  BRCA_Mutation  \
0               0             10                      3  ...              0   
1               0              7                      4  ...              0   
2               0              4                      1  ...              0   

   H_Pylori_Infection  Calcium_Intake  Overall_Risk_Score   BMI  \
0                   0               0            0.601460  27.2   
1                   0               3            0.380986  28.7   
2                   0               7            0.265377  26.8   

   Physical_Activity_Level  Risk_Level  county_STATE   county_CTYN

In [10]:
# --- VERIFICAR VALORES NULOS ---
print("\n--- VALORES NULOS ---")
print(df.isnull().sum())

# --- SEPARAR CARACTERÍSTICAS (X) Y VARIABLE OBJETIVO (y) ---
X = df.drop('Risk_Level', axis=1)
y = df['Risk_Level']

print("\n--- DISTRIBUCIÓN DE LA VARIABLE OBJETIVO ---")
print(y.value_counts())


--- VALORES NULOS ---
Patient_ID                 0
Cancer_Type                0
Age                        0
Gender                     0
Smoking                    0
Alcohol_Use                0
Obesity                    0
Family_History             0
Diet_Red_Meat              0
Diet_Salted_Processed      0
Fruit_Veg_Intake           0
Physical_Activity          0
Air_Pollution              0
Occupational_Hazards       0
BRCA_Mutation              0
H_Pylori_Infection         0
Calcium_Intake             0
Overall_Risk_Score         0
BMI                        0
Physical_Activity_Level    0
Risk_Level                 0
county_STATE               0
county_CTYNAME             0
county_POPESTIMATE2015     0
dtype: int64

--- DISTRIBUCIÓN DE LA VARIABLE OBJETIVO ---
Risk_Level
Medium    2368
Low        484
High       148
Name: count, dtype: int64


In [11]:
# --- IDENTIFICAR TIPOS DE COLUMNAS ---
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"\n--- COLUMNAS NUMÉRICAS ({len(numeric_features)}): ---")
print(numeric_features)
print(f"\n--- COLUMNAS CATEGÓRICAS ({len(categorical_features)}): ---")
print(categorical_features)


--- COLUMNAS NUMÉRICAS (21): ---
['Patient_ID', 'Age', 'Gender', 'Smoking', 'Alcohol_Use', 'Obesity', 'Family_History', 'Diet_Red_Meat', 'Diet_Salted_Processed', 'Fruit_Veg_Intake', 'Physical_Activity', 'Air_Pollution', 'Occupational_Hazards', 'BRCA_Mutation', 'H_Pylori_Infection', 'Calcium_Intake', 'Overall_Risk_Score', 'BMI', 'Physical_Activity_Level', 'county_STATE', 'county_POPESTIMATE2015']

--- COLUMNAS CATEGÓRICAS (2): ---
['Cancer_Type', 'county_CTYNAME']


In [12]:
# --- CREAR LOS TRANSFORMADORES (PREPROCESAMIENTO) ---
numeric_transformer = StandardScaler()

# Transformador para datos categóricos: One-Hot Encoding (crear columnas dummy)
categorical_transformer = OneHotEncoder(drop='first', handle_unknown='ignore') # drop='first' evita multicolinealidad

# Combinamos ambos transformadores en un solo preprocesador
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [13]:

# --- GUARDAR LOS DATOS PREPROCESADOS ---
# Guardamos los arrays de NumPy para que el equipo pueda usarlos directamente sin volver a ejecutar todo el preprocesamiento.
# Esto es más eficiente para el modelado.
import os
output_dir = r'/content/Untitled Folder 1'

np.save(os.path.join(output_dir, 'X_train_preprocesado.npy'), X_train)
np.save(os.path.join(output_dir, 'X_test_preprocesado.npy'), X_test)
np.save(os.path.join(output_dir, 'y_train_preprocesado.npy'), y_train)
np.save(os.path.join(output_dir, 'y_test_preprocesado.npy'), y_test)

print("\n--- ARCHIVOS PREPROCESADOS GUARDADOS ---")
print("1. X_train_preprocesado.npy")
print("2. X_test_preprocesado.npy")
print("3. y_train_preprocesado.npy")
print("4. y_test_preprocesado.npy")


--- ARCHIVOS PREPROCESADOS GUARDADOS ---
1. X_train_preprocesado.npy
2. X_test_preprocesado.npy
3. y_train_preprocesado.npy
4. y_test_preprocesado.npy


In [14]:
# --- APLICAR EL PREPROCESAMIENTO ---
# Ajustamos el preprocesador a los datos y los transformamos
# Esto convertirá nuestras características (X) en un array de numpy listo para modelar
X_preprocesado = preprocessor.fit_transform(X)

print("\n--- PREPROCESAMIENTO COMPLETADO ---")
print(f"Forma de los datos preprocesados: {X_preprocesado.shape}")

# --- VOLVER A DIVIDIR EN TRAIN/TEST ---
# Aunque ya tienen archivos train.csv y test.csv, es mejor hacer la división DESPUÉS del preprocesamiento.
# Usaremos una semilla (random_state) para que los resultados sean reproducibles.
X_train, X_test, y_train, y_test = train_test_split(X_preprocesado, y, test_size=0.2, random_state=42, stratify=y)

print(f"\n--- DIVISIÓN TRAIN/TEST ---")
print(f"Tamaño de entrenamiento (X_train): {X_train.shape}")
print(f"Tamaño de prueba (X_test): {X_test.shape}")


--- PREPROCESAMIENTO COMPLETADO ---
Forma de los datos preprocesados: (3000, 611)

--- DIVISIÓN TRAIN/TEST ---
Tamaño de entrenamiento (X_train): (2400, 611)
Tamaño de prueba (X_test): (600, 611)


In [15]:
# --- GUARDAR EL PREPROCESADOR ---
import joblib
joblib.dump(preprocessor, os.path.join(output_dir, 'preprocessor.pkl'))

print("\n--- PREPROCESADOR GUARDADO ---")
print("Archivo: preprocessor.pkl")


--- PREPROCESADOR GUARDADO ---
Archivo: preprocessor.pkl


In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# --- CARGAR LOS DATOS LIMPIOS ---
df = pd.read_csv("Dataset_ALDIMI_Merged_Clean.csv")

print("--- DATOS ORIGINALES ---")
print(f"Forma del dataset: {df.shape}")
print(df.head(3))
print("\nInformación del dataset:")
df.info()

# --- VERIFICAR VALORES NULOS ---
print("\n--- VALORES NULOS ---")
print(df.isnull().sum())

# --- SEPARAR CARACTERÍSTICAS (X) Y VARIABLE OBJETIVO (y) ---
X = df.drop('Risk_Level', axis=1)
y = df['Risk_Level']

print("\n--- DISTRIBUCIÓN DE LA VARIABLE OBJETIVO ---")
print(y.value_counts())

# --- IDENTIFICAR TIPOS DE COLUMNAS ---
# Identificamos columnas numéricas y categóricas para aplicarles diferentes transformaciones
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"\n--- COLUMNAS NUMÉRICAS ({len(numeric_features)}): ---")
print(numeric_features)
print(f"\n--- COLUMNAS CATEGÓRICAS ({len(categorical_features)}): ---")
print(categorical_features)

# --- CREAR LOS TRANSFORMADORES (PREPROCESAMIENTO) ---
# Transformador para datos numéricos: Estandarizar (media = 0, desviación = 1)
numeric_transformer = StandardScaler()

# Transformador para datos categóricos: One-Hot Encoding (crear columnas dummy)
categorical_transformer = OneHotEncoder(drop='first', handle_unknown='ignore') # drop='first' evita multicolinealidad

# Combinamos ambos transformadores en un solo preprocesador
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# --- APLICAR EL PREPROCESAMIENTO ---
# Ajustamos el preprocesador a los datos y los transformamos
# Esto convertirá nuestras características (X) en un array de numpy listo para modelar
X_preprocesado = preprocessor.fit_transform(X)

print("\n--- PREPROCESAMIENTO COMPLETADO ---")
print(f"Forma de los datos preprocesados: {X_preprocesado.shape}")

# --- VOLVER A DIVIDIR EN TRAIN/TEST ---
# Aunque ya tienen archivos train.csv y test.csv, es mejor hacer la división DESPUÉS del preprocesamiento.
# Usaremos una semilla (random_state) para que los resultados sean reproducibles.
X_train, X_test, y_train, y_test = train_test_split(X_preprocesado, y, test_size=0.2, random_state=42, stratify=y)

print(f"\n--- DIVISIÓN TRAIN/TEST ---")
print(f"Tamaño de entrenamiento (X_train): {X_train.shape}")
print(f"Tamaño de prueba (X_test): {X_test.shape}")

# --- GUARDAR LOS DATOS PREPROCESADOS ---
# Guardamos los arrays de NumPy para que el equipo pueda usarlos directamente sin volver a ejecutar todo el preprocesamiento.
# Esto es más eficiente para el modelado.
import os
output_dir = r'/content/Untitled Folder 1'

np.save(os.path.join(output_dir, 'X_train_preprocesado.npy'), X_train)
np.save(os.path.join(output_dir, 'X_test_preprocesado.npy'), X_test)
np.save(os.path.join(output_dir, 'y_train_preprocesado.npy'), y_train)
np.save(os.path.join(output_dir, 'y_test_preprocesado.npy'), y_test)

print("\n--- ARCHIVOS PREPROCESADOS GUARDADOS ---")
print("1. X_train_preprocesado.npy")
print("2. X_test_preprocesado.npy")
print("3. y_train_preprocesado.npy")
print("4. y_test_preprocesado.npy")

# --- (MUY IMPORTANTE) PASO 10: GUARDAR EL PREPROCESADOR ---
# Guardamos el objeto 'preprocessor' para que, cuando el modelo esté en producción, podamos transformar NUEVOS datos exactamente de la misma forma.
import joblib
joblib.dump(preprocessor, os.path.join(output_dir, 'preprocessor.pkl'))

print("\n--- PREPROCESADOR GUARDADO ---")
print("Archivo: preprocessor.pkl")

--- DATOS ORIGINALES ---
Forma del dataset: (3000, 24)
   Patient_ID Cancer_Type  Age  Gender  Smoking  Alcohol_Use  Obesity  \
0        1001        Lung   52       0       10            7        4   
1        1002       Colon   73       1        5            4        8   
2        1003        Skin   70       0        3            0        2   

   Family_History  Diet_Red_Meat  Diet_Salted_Processed  ...  BRCA_Mutation  \
0               0             10                      3  ...              0   
1               0              7                      4  ...              0   
2               0              4                      1  ...              0   

   H_Pylori_Infection  Calcium_Intake  Overall_Risk_Score   BMI  \
0                   0               0            0.601460  27.2   
1                   0               3            0.380986  28.7   
2                   0               7            0.265377  26.8   

   Physical_Activity_Level  Risk_Level  county_STATE   county_CTYN